In [ ]:
"""
Week 4 - Day 2
Business Dashboard
==================
Complete business dashboard showing
all project results and proofs.

KEY notebook for Final Review!

Infotact DS/ML Internship — Project 2
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1] / "src"
sys.path.insert(0, str(PROJECT_ROOT))

from environment.pricing_env import DynamicPricingEnv
from agents.ppo.ppo_agent import PPOAgent
from agents.dqn.dqn_agent import DQNAgent
from agents.q_learning_agent import (
    QLearningAgent, QL_CONFIG
)
from agents.baseline_agents import (
    FixedPriceAgent, TimedPricingAgent,
    DemandBasedAgent, LinearDecayAgent
)
from simulation.final_simulation import (
    run_final_simulation,
    run_statistical_proof
)
from simulation.business_value import (
    calculate_business_value
)
from visualization.business_dashboard import (
    create_business_dashboard
)
from analysis.trajectory_insights_final import (
    prove_all_behaviors
)
from training.config_manager import (
    BEST_PPO_CONFIG, BEST_DQN_CONFIG
)

plt.style.use('seaborn-v0_8')
print("✅ Business dashboard modules loaded!")
print("\nThis is the KEY notebook for Final Review!")

In [ ]:
env = DynamicPricingEnv()

print("Training all agents...\n")

ppo = PPOAgent(env, BEST_PPO_CONFIG)
ppo.train(n_episodes=2000, verbose=False)
print("✅ PPO ready!")

dqn = DQNAgent(env, BEST_DQN_CONFIG)
dqn.train(n_episodes=2000, verbose=False)
print("✅ DQN ready!")

ql = QLearningAgent(env, QL_CONFIG)
ql.train(n_episodes=3000, verbose=False)
print("✅ Q-Learning ready!")

agents = {
    'Fixed Price'  : FixedPriceAgent(env),
    'Time Based'   : TimedPricingAgent(env),
    'Demand Based' : DemandBasedAgent(env),
    'Linear Decay' : LinearDecayAgent(env),
    'Q-Learning'   : ql,
    'DQN'          : dqn,
    'PPO'          : ppo,
}
print(f"\n✅ All {len(agents)} agents ready!")

In [ ]:
print("Running 1000-season simulation...")
all_results, summary_df = run_final_simulation(
    agents, env, n_seasons=1000
)
print("\n✅ Simulation complete!")

In [ ]:
dashboard_metrics = create_business_dashboard(
    summary_df, all_results, ppo, env,
    save_path='../results/business_dashboard.png'
)
print("\n✅ Main dashboard created!")

In [ ]:
behavior_proof = prove_all_behaviors(
    ppo, env, n_episodes=200,
    save_path='../results/behavior_proof.png'
)

In [ ]:
proof = run_statistical_proof(
    all_results, winner='PPO'
)
sig_count = sum(
    1 for v in proof.values()
    if v.get('significant', False)
)
print(f"\n✅ PPO statistically better than "
      f"{sig_count}/{len(proof)} agents!")

In [ ]:
bv = calculate_business_value(
    summary_df, winner='PPO'
)
print(f"\n📊 Business Value Summary:")
print(f"  Daily uplift  : +${bv['daily_uplift']:.0f}")
print(f"  Monthly uplift: +${bv['monthly_uplift']:.0f}")
print(f"  Annual uplift : +${bv['annual_uplift']:.0f}")

In [ ]:
ppo_rev = summary_df[
    summary_df['Agent'] == 'PPO'
]['Mean Revenue'].values[0]

best_bl = summary_df[
    summary_df['Agent'].isin([
        'Fixed Price', 'Time Based',
        'Demand Based', 'Linear Decay'
    ])
]['Mean Revenue'].max()

imp = (ppo_rev - best_bl) / best_bl * 100
medals = ['🥇', '🥈', '🥉',
          '4️⃣', '5️⃣', '6️⃣', '7️⃣']

print("╔══════════════════════════════════════════╗")
print("║   WEEK 4 DAY 2 — DASHBOARD COMPLETE!    ║")
print("╠══════════════════════════════════════════╣")
print("║  FINAL RANKINGS (1000 Seasons):          ║")
for i, row in summary_df.iterrows():
    print(f"║  {medals[i]} {row['Agent']:<18}: "
          f"${row['Mean Revenue']:<8.0f}     ║")
print("╠══════════════════════════════════════════╣")
print("║  BUSINESS VALUE:                         ║")
print(f"║  Daily Uplift  : +${bv['daily_uplift']:.0f}"
      f"{'':<21} ║")
print(f"║  Annual Uplift : +${bv['annual_uplift']:.0f}"
      f"{'':<18} ║")
print(f"║  Improvement   : +{imp:.1f}%"
      f"{'':<23} ║")
print("╠══════════════════════════════════════════╣")
print("║  BEHAVIORS PROVED:                       ║")
print(f"║  Deadline drop : "
      f"-{behavior_proof['deadline_drop_pct']:.1f}%"
      f"{'':<22} ║")
print(f"║  Scarcity prem : "
      f"+{behavior_proof['scarcity_premium_pct']:.1f}%"
      f"{'':<22} ║")
print("╠══════════════════════════════════════════╣")
print("║  Tomorrow → Final Documentation 📝       ║")
print("╚══════════════════════════════════════════╝")